In [ ]:
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText
import os
from datetime import datetime
from dotenv import load_dotenv
from huggingface_hub import login
import pandas as pd

load_dotenv()
hf_token = os.getenv("HF_TOKEN")
login(token=hf_token)

In [ ]:
def parse_in_json(llm_response):
    temp = eval(llm_response)
    if not isinstance(temp, dict):
        return {
            "time": 0.0,
            "coordinate": [[0, 0], [0, 0]],
            "type": "head-on",
            "why": "invalid response format"
        }
    return temp

def save_submission(submission, experiment_name):
    timestamp = datetime.now().strftime("%Y%m%d%H%M%S")
    save_dir = os.path.join("result", f"{experiment_name}_{timestamp}")
    os.makedirs(save_dir, exist_ok=True)
    submission_path = os.path.join(save_dir, "submission.csv")
    description_path = os.path.join(save_dir, "description.txt")
    submission.to_csv(submission_path, index=False, lineterminator="\n")
    open(description_path, "w").close()
    print(f"saved to {save_dir}")

def make_submission(results):
    rows = []
    for result in results:
        video_path = result.get("video_path", "")
        path = "videos/" + video_path.split("/videos/")[-1] if "/videos/" in video_path else video_path
        accident_time = float(result.get("time", 0.0))
        coordinate = result.get("coordinate", [[0, 0], [0, 0]])
        (x1, y1), (x2, y2) = coordinate
        center_x = round(((x1 + x2) / 2) / 1000, 3)
        center_y = round(((y1 + y2) / 2) / 1000, 3)
        accident_type = result.get("type", "unknown")
        rows.append({
            "path": path,
            "accident_time": round(accident_time, 2),
            "center_x": center_x,
            "center_y": center_y,
            "type": accident_type
        })

    submission = pd.DataFrame(rows, columns=["path", "accident_time", "center_x", "center_y", "type"])
    return submission

In [ ]:
class VideoInferenceVLM:
    def video_inference(self, video_path, prompt, max_new_tokens=128):
        raise NotImplementedError("Subclasses should implement this method.")

class Qwen3VLInference(VideoInferenceVLM):
    def __init__(self, model_id = "Qwen/Qwen3-VL-8b-Instruct"):

        self.model = AutoModelForImageTextToText.from_pretrained(
            model_id, dtype="auto", device_map="auto"
        )
        self.processor = AutoProcessor.from_pretrained(model_id)

    def video_inference(self, video_path, prompt, max_new_tokens=128):
        # Messages containing a video url(or a local path) and a text query
        messages = [
            {
                "role": "user",
                "content": [
                    {
                        "type": "video",
                        "video": video_path,
                    },
                    {"type": "text", "text": prompt},
                ],
            }
        ]
        # Preparation for inference
        inputs = self.processor.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_dict=True,
            return_tensors="pt"
        )
        inputs = inputs.to(self.model.device)

        # Inference: Generation of the output
        with torch.inference_mode():
            generated_ids = self.model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, use_cache=True)

        generated_ids_trimmed = [
            out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
        ]
        output_text = self.processor.batch_decode(
            generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
        )
        return output_text

inference = Qwen3VLInference()

In [ ]:
instruction = "The video is represented as indexed frames in chronological order. " \
    "Find the first indexed frame where physical contact between vehicles begins. " \
    "Return that frame index first. " \
    "Then return the corresponding time for that indexed frame. " \
    "Also return one collision bounding box on that indexed frame using left-top and right-bottom coordinates. " \
    "The bounding box should enclose the collision region or the involved vehicles at the first contact moment. " \
    "Then return the collision type. The collision type includes: head-on, rear-end, sideswipe, single, and t-bone collisions. " \
    "Head-on is defined as a collision where the front ends of two vehicles hit each other. " \
    "Rear-end is defined as a collision where the front end of one vehicle hits the rear end of another vehicle. " \
    "Sideswipe is defined as a slight collision where the sides of two vehicles hit each other. " \
    "Single is defined as an accident that involves only one vehicle, such as a vehicle hitting a stationary object or a vehicle losing control and crashing without colliding with another vehicle. " \
    "T-bone is defined as a collision where the front end of one vehicle hits the side of another vehicle, forming a 'T' shape. " \
    "Important: first collision means the earliest frame where actual physical contact starts, not before and not the aftermath. " \
    "Do not default to frame 0. Search the indexed frames step by step in time order."
return_format = """
please return the result in JSON format only, not markdown.
here is the JSON format:
{
    "frame_index": exact indexed frame where the first collision occurs,
    "time": exact time corresponding to that indexed frame,
    "coordinate": [
        [x1, y1],
        [x2, y2]
    ],
    "type": "choose one from [head-on, rear-end, sideswipe, single, t-bone]",
    "why": "explain why this indexed frame is the first collision moment, why this bounding box is correct, and why this collision type matches."
}
---
example:
{
    "frame_index": int,
    "time": "03.27",
    "coordinate": [
        [x1, y1],
        [x2, y2]
    ],
    "type": "choose one from [head-on, rear-end, sideswipe, single, t-bone]",
    "why": "Frame *frame* is the earliest indexed frame where ... The bounding box *[x1, y1], [x2, y2]* is located at...  This is *type* because ..."
}
"""

# video_path = "/workspace/dataset/videos/__WFqm4i3vE_00.mp4"
# output = inference.video_inference(video_path, instruction + return_format, max_new_tokens=128)
# print(output[0])

In [ ]:
test_path_file = "dataset/test_video_path.txt"
prompt = instruction + return_format
results = []

with open(test_path_file, "r") as f:
    video_paths = [line.strip() for line in f if line.strip()]

for i, video_path in enumerate(video_paths):
    output = inference.video_inference(
        video_path,
        prompt,
        max_new_tokens=128
    )
    output_json = parse_in_json(output[0])
    output_json["video_path"] = video_path
    results.append(output_json)
    print(f"{i+1}/{len(video_paths)} is done.")

In [ ]:
submission = make_submission(results)
save_submission(submission, "qwen3_vl_experiment")